# Stimulus-programme rollout tool

Give it a **stimulus trajectory file** and a **trained model**; get simulated fish trajectories and an mp4.
No training, no real fish, no `PositionTxt` needed.

The trajectory file is the thing the Bonsai rig actually plays: one row per frame at 30 fps, 8 dots x
`(name, x, y, size, color, backg)`, with `x`/`y` in **millimetres relative to the fish** (left = +x).
It is fish-agnostic — `CLsemi` means the dot track is anchored to each fish's pose at block onset and
then replayed in that latched frame.

Here that fish is the **simulated** one, so this is a genuine semi-closed-loop simulation rather than a
replay: the same file produces a different absolute dot track on every rollout.

**Input contract for a new file** — keep the 48-column `name,x,y,size,color,backg` layout (8 dot slots,
pad unused ones with `none,0,0,0,1,0`) and the `CLsemi..._g{L}-{R}_s{sL}-{sR}` label grammar; use `none`
for pauses. Anything not starting with `CLsemi` is treated as no stimulus.

Cost: rollouts take **seconds** (~3 s per 175-min fish). The **mp4 is the slow step** — minutes — so it is
rendered in a subprocess and written to disk, never displayed inline.

In [ ]:
# =============================== CONFIG ==================================== #
import sys, time, json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists() and (ROOT.parent / 'scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
if not hasattr(np, '_core'):                 # load Colab (numpy 2) checkpoints under numpy 1
    import numpy.core as _nc; np._core = _nc

def pick(*candidates):
    """First path that exists, else None. Everything the tool reads is resolved this way:
    the full local dataset when you have it, the small shipped assets/ copy otherwise."""
    for c in candidates:
        p = ROOT / c
        if p.exists():
            return p
    return None

# --- inputs (local full data preferred, shipped assets as fallback) --------- #
TRAJ_FILE = pick('data/data/Original_data/2025.02.20-0957_trajectory_GsizeMotionSingle_180min.csv',
                 'assets/stimulus/2025.02.20-0957_trajectory_GsizeMotionSingle_180min.csv.gz')
ROI_FILE  = pick('data/data/Original_data/ROIdef2025-02-20T12_09_16.csv',
                 'assets/ROIdef2025-02-20T12_09_16.csv')
BOUTS     = pick('data/bouts_GsizeMotionSingle.csv')      # None in a clean clone -- that is fine
REFERENCE = pick('assets/real_reference.npz')

# --- knobs ----------------------------------------------------------------- #
MODEL_KEY  = 'J10'          # see the registry in the next cell
N_ROLLOUTS = 3              # the video shows the first 3
SEED       = 0
FISH_MODE  = 'new'          # 'new' = fresh synthetic fish; 'fish:7' = start from real fish 7's pose
WINDOWS    = None           # None = auto-pick two contrasting blocks; else [(lo, hi, label), ...]
VIDEO_SPEED = 3.0           # 3x real time, as the existing comparison videos use
OUT_DIR    = ROOT / 'tool_videos'

plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': 'white', 'savefig.facecolor': 'white', 'figure.dpi': 110})
REAL, GEN = '#3C3C3C', '#1F3B8C'
print('ROOT     ', ROOT)
print('python   ', sys.version.split()[0], '| numpy', np.__version__)
print('stimulus ', TRAJ_FILE.name if TRAJ_FILE else 'MISSING')
print('roi      ', ROI_FILE.name if ROI_FILE else 'MISSING')
print('reference', 'bouts CSV (full local data)' if BOUTS else
                   (REFERENCE.name if REFERENCE else 'MISSING'))
assert TRAJ_FILE and ROI_FILE and (BOUTS or REFERENCE), 'missing inputs -- see the README'

In [ ]:
# ==================== MODEL REGISTRY + REAL REFERENCE ====================== #
# Joint-family checkpoints only: programme rollouts need the stim_provider hook in
# fit_MLP_joint.generate_mix_joint_loop, which the J10/J17 lineage shares.
import evaluate as E, fit_MLP_border as MB, fit_MLP_mix as MX
import stim_program as SP, stim_rollout as SR, tool_video as TV, tool_reference as TR

# key -> candidate paths, first existing wins. Only J10 ships with the repo; the rest resolve
# just for whoever has the full research checkout.
MODELS = {
    'J10':           ['assets/models/ladder_b_indiv.pt',        # the thesis J10 (traj AUC 0.704)
                      'notebooks/_zip_full/ladder_b_indiv.pt'],
    'J10_150ep':     ['ladder_b_indiv150.pt'],                  # same recipe, 150 epochs
    'J10_cal2q':     ['ladder_b_indiv150_cal2q.pt'],            # recalibrated containment
    'J10_psi_basis': ['ladder_b_indiv_psib_cal2q.pt'],          # J10 + psi-frame + border basis
    'J17':           ['ladder_re_swap_gr.pt'],                  # fitted per-fish latent, m=2
}
_p = pick(*MODELS[MODEL_KEY])
if _p is None:
    raise FileNotFoundError(
        f"{MODEL_KEY} not found. Tried: {MODELS[MODEL_KEY]}. "
        f"Only 'J10' ships with the repo; the others need the full research checkout.")
model = MX.load_mix(str(_p))
cf = model._cfg_feat
print(f'{MODEL_KEY}: {model.name}  ({_p.relative_to(ROOT)})')
print(f"  stim_mode={cf['stim_mode']}  border={cf.get('border_mode')}  wall={cf.get('wall_mode')}"
      f"  psi_ref={cf.get('psi_ref', 'ori')}  dt_snap={cf.get('dt_snap', False)}")
print(f"  closed_loop={model.closed_loop}  stim_loop={model.stim_loop}  "
      f"contain={cf.get('centre_bias') is not None}  indiv={cf.get('indiv') is not None}")

# The real-data reference: start-pose occupancy + the comparison curves for the sanity panel.
# ONE code path either way -- with the bouts CSV present the summary is recomputed in memory, so
# a clean clone and a full checkout draw identical curves and sample identical occupancy.
records = MB.build_border_records(E.load_bouts(str(BOUTS)), roi_path=str(ROI_FILE)) if BOUTS else None
ref = TR.reference_from_records(records) if records else TR.load_reference(REFERENCE)
print()
print(ref.summary())

In [ ]:
# ============================ THE PROGRAMME ================================ #
program = SP.load_program(TRAJ_FILE)      # .csv or .csv.gz, pandas handles both
print(program.summary())

# Acceptance test for the anchoring transform. Only possible when the RECORDED experiment that
# matches this programme is on disk -- for a genuinely new file, or a clean clone, there is
# nothing to compare against and it skips.
try:
    SP.verify_against_transformed(program, transformed_dir=ROOT / 'data/data/df_transformed')
except (FileNotFoundError, RuntimeError) as err:
    print(f'(anchoring check skipped: {err})')

program.block_table().head(10)

In [ ]:
# ============================== ROLL OUT =================================== #
dish = SR.default_dish(roi_path=str(ROI_FILE))
if FISH_MODE == 'new':
    # Fresh synthetic fish: start pose drawn from the real occupancy histogram, and J10's
    # per-trajectory individuality (indiv/vigor) drawn fresh. Rollout k is individual k.
    start_poses, sampler = None, ref.sample_start_pose
elif records is not None:
    fid = int(FISH_MODE.split(':')[1])           # 'fish:7' -- needs the full local data
    r = records[fid]
    dish = (r.dish_cx, r.dish_cy, r.dish_R)
    start_poses, sampler = [(float(r.x_px[0]), float(r.y_px[0]), float(r.f_ori_rad[0]))] * N_ROLLOUTS, None
    print(f'starting every rollout from real fish {fid}: dish {dish}')
else:
    raise RuntimeError("FISH_MODE='fish:N' needs data/bouts_GsizeMotionSingle.csv (not shipped).")

t0 = time.time()
rollouts = SR.rollout_many(model, program, N_ROLLOUTS, seed=SEED,
                           dish=dish, start_poses=start_poses, pose_sampler=sampler)
print(f'{N_ROLLOUTS} rollouts in {time.time() - t0:.1f}s')

## How the fish sees the stimulus

The model is **bout-level**, so the programme is read **once per bout onset**, at
`frame = round(sum(dt) * 30)`. Four consequences worth being explicit about:

1. **The fish is blind between bouts** (~1.3 s). That is a property of the existing model, not of this
   tool — the ordinary rollout also carries one stimulus row per bout. What the tool changes is *where
   the value comes from*, not how often it is read.
2. **A block change is noticed at the first bout at or after the onset frame**, not at the frame itself.
   That is physically right: the fish can only act at a bout.
3. **The dot anchor is still exact.** Under the dart-and-hold reconstruction the fish is stationary from
   `onset(t-1) + 3` frames until `onset(t)`, so its pose at the true block-onset frame is identical to the
   pose the provider is handed — even if a rare long interval swallows most of a block. The only slack is
   an onset landing inside the 3-frame dart (<= 0.1 s), plus bounce bouts, where the guardrail leaves the
   heading reflected rather than along the realised chord (a few percent of bouts).
4. **The run ends at the last bout whose onset is still inside the programme.** The bout that would start
   past the end is never generated, so every emitted bout was driven by a real stimulus condition — and
   the trajectory *undershoots* the programme by up to one inter-bout interval. The next cell prints the
   realised numbers rather than asserting them.

In [ ]:
rows = []
for k, g in enumerate(rollouts):
    cx, cy, R = g['dish']
    rr = np.hypot(g['x_px'] - cx, g['y_px'] - cy) / R
    rows.append(dict(rollout=k, bouts=len(g['dtheta']),
                     last_onset_min=g['last_onset_s'] / 60,
                     short_by_s=g['program_s'] - g['last_onset_s'],
                     final_dt_s=float(g['dt_s'][-1]),
                     median_dt_s=float(np.median(g['dt_s'])),
                     mean_rR=float(rr.mean()), max_rR=float(rr.max()),
                     outside_dish=int((rr > 1.0).sum()), wall_s=g['wall_s']))
clock = pd.DataFrame(rows)
print(f"programme {program.duration_s / 60:.2f} min; "
      f"every rollout short by <= its own final interval: "
      f"{bool((clock.short_by_s <= clock.final_dt_s + 1e-6).all())}")
clock.round(3)

In [ ]:
# ======================= IS IT ACTUALLY REACTING? ========================== #
# Generated (blue) needs only the rollouts. The real (black) reference is a POPULATION BASELINE
# from the recorded experiment -- reusable for any programme whose conditions stay in the recorded
# vocabulary (N in 1,2,4,6,8; moving/static). It is a baseline, not a matched control: if your new
# programme changes the MIX of conditions, the three marginal histograms shift for compositional
# reasons too. The following index by N is robust to that; it is also the only panel that would
# collapse (flat at 0.5) if the stimulus provider were mis-wired.
c = ref.curves

gen_d  = np.concatenate([g['dtheta'] for g in rollouts])
gen_p  = np.concatenate([g['stim_angle_gen'] for g in rollouts])
gen_dt = np.concatenate([g['dt_s'] for g in rollouts])
gen_rr = np.concatenate([np.hypot(g['x_px'] - g['dish'][0], g['y_px'] - g['dish'][1]) / g['dish'][2]
                         for g in rollouts])
gen_n  = np.concatenate([program.n_dots_of[g['block_id']] for g in rollouts])
gen_mv = np.concatenate([program.moving_of[g['block_id']].astype(float) for g in rollouts])

fi_real = c['following_by_n']
fi_gen  = TR.following_by_n(gen_d, gen_p, gen_n, gen_mv)      # same function both sides

fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))
ax[0].plot(list(fi_real), list(fi_real.values()), 'o-', color=REAL, label='real')
ax[0].plot(list(fi_gen), list(fi_gen.values()), 's-', color=GEN, label=MODEL_KEY)
ax[0].axhline(0.5, color='0.7', lw=0.8, ls='--')
ax[0].set(xlabel='group size N', ylabel='P(turn toward)', title='following index by N')
ax[0].legend(frameon=False)
for a, (key, gv, lab) in zip(ax[1:], [('abs_dtheta', np.abs(gen_d), '|dtheta| (deg)'),
                                      ('dt', gen_dt, 'dt (s)'),
                                      ('rr', gen_rr, 'r / R')]):
    dens, edges = c[key]
    a.step(edges[:-1], dens, where='post', color=REAL, label='real')
    a.hist(gv[np.isfinite(gv)], bins=len(dens), range=(edges[0], edges[-1]),
           density=True, histtype='step', color=GEN, label=MODEL_KEY)
    a.set(xlabel=lab); a.legend(frameon=False)
plt.tight_layout(); plt.show()

print('following index  real:', {k: round(v, 3) for k, v in fi_real.items()})
print('                 gen :', {k: round(v, 3) for k, v in fi_gen.items()})
print(f"mean r/R  real {c['mean_rR']:.3f}  gen {gen_rr.mean():.3f}   "
      f'| frames outside dish: {int((gen_rr > 1.0).sum())}')

In [ ]:
# ============================ VIDEO WINDOWS ================================ #
# Default: the first MOVING block at the largest group size, and the first STATIC block at the
# same size -- matched N, so the only thing differing between the two rows is shoal motion.
windows = WINDOWS if WINDOWS is not None else TV.pick_windows(program)
for i, (lo, hi, lab) in enumerate(windows):
    print(f'row {i}: {lab}  frames {lo}-{hi}  ({(hi - lo) / program.fps:.0f}s)')

In [ ]:
# ================================ RENDER =================================== #
# Rows = the two windows, columns = the first 3 rollouts (same fish across both rows).
# Runs in a subprocess: a 6-panel FuncAnimation over ~1800 frames can take down the kernel, and
# the child also picks up the bundled-ffmpeg shim so .mp4 works with no system ffmpeg.
# THIS IS THE SLOW CELL -- minutes, unlike the seconds the rollouts take.
OUT_DIR.mkdir(parents=True, exist_ok=True)
# .stem would leave a trailing '.csv' on the shipped '.csv.gz' asset, so strip both suffixes.
stem = TRAJ_FILE.name.removesuffix('.gz').removesuffix('.csv')
out_mp4 = OUT_DIR / f'{stem}__{MODEL_KEY}.mp4'

t0 = time.time()
out_mp4, manifest = TV.make_tool_video(rollouts, program, out_mp4,
                                       windows=windows, n_cols=3, speed=VIDEO_SPEED)
print(f'\nrendered in {time.time() - t0:.0f}s -> {out_mp4}')
print(f'{manifest["n_frames"]} frames/panel at {manifest["fps"]} fps x {VIDEO_SPEED} speed')

## Using it on something else

**A new stimulus file** — point `TRAJ_FILE` at it and re-run. Nothing else changes: block structure,
group sizes, motion and durations are all read from the file. The anchoring check in the programme cell
will skip itself (there is no recorded experiment to compare against), which is expected.

**A different model** — change `MODEL_KEY` to any entry in `MODELS`. It must be a joint-family checkpoint
(J10/J17 lineage); anything else raises rather than silently teacher-forcing the stimulus.

**More fish** — raise `N_ROLLOUTS`. The video always shows the first 3, but the sanity panel uses all of
them, so 10–15 gives much tighter following-index estimates for ~3 s each.

**A specific individual** — `FISH_MODE = 'fish:7'` starts every rollout from real fish 7's first pose and
dish. Note this reproduces its *arena and starting point*, not its fitted identity: J10's individuality is
drawn per trajectory, so the fish are still fresh draws from the population.

**Other windows** — set `WINDOWS = [(lo, hi, 'label'), (lo2, hi2, 'label2')]` in frames.

**Faster video** — shorten the windows (`TV.pick_windows(program, n_frames=600)` for 20 s), raise
`VIDEO_SPEED`, or change the output extension to `.gif` for a quick look.